# DiagramIO: đo hiệu năng pha 2

Notebook chạy bộ đo trong thư mục `bench/` của repo DiagramIO (xem `bench/README.md` và `docs/adr/0001-phuong-phap-do.md`).

- **Settings**: Accelerator **None** (CPU), bật **Internet**.
- Chạy nền để không phụ thuộc trình duyệt: **Save Version → Save & Run All (Commit)**. Nên chạy `quick` trước (khoảng 30 phút) để kiểm tra môi trường.
- Ma trận `b4` (mặc định, tag `v2.4-b4`): một app (C3) so với 1, 2, 3 instance sau Nginx (C4x1, C4x2, C4x3), kèm W7 (một instance chết đột ngột) và W7r (khởi động lại cuốn chiếu); khoảng 4,5 giờ với `REPS = 2`.
- Ma trận cũ, để chạy lại: `b3` (cache token của B3, tag `v2.3.2`, khoảng 3,5 giờ), `b12` (C0, C1, C2, C3, khoảng 4,7 giờ với `REPS = 2`), `tune` (tinh chỉnh runtime của B1), `m0` (chỉ C0).
- Kết quả: thư mục `DiagramIO/bench/results/<phiên>/` và file `results.tgz` trong tab **Output**. Giải nén vào `bench/results/` của repo rồi commit các file tóm tắt.

In [ ]:
REPO_URL = "https://github.com/vietsosad/DiagramIO_16.git"
REF = "v2.4-b4"   # commit hoặc tag chứa thư mục bench/; cấu hình không có tag riêng được build từ REF
MATRIX = "b4"     # "quick" để chạy thử; "b3", "b12", "tune", "m0" là các ma trận cũ
REPS = 2          # số lần lặp mỗi phép đo

In [ ]:
import os
import subprocess

os.chdir("/kaggle/working")
if not os.path.isdir("DiagramIO"):
    subprocess.run(["git", "clone", REPO_URL, "DiagramIO"], check=True)
os.chdir("/kaggle/working/DiagramIO")
subprocess.run(["git", "fetch", "--tags", "origin"], check=True)
subprocess.run(["git", "checkout", "--detach", REF], check=True)
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
# Cài PostgreSQL 16, Redis 7.4, Nginx, JDK 21, k6, async-profiler và cấu hình PostgreSQL (khoảng 3-4 phút)
!bash bench/kaggle/setup.sh

In [ ]:
# Build jar từng cấu hình (cấu hình có tag riêng như C0, C1 của b12 build từ tag đó; còn lại từ REF), sinh và nạp
# dữ liệu, EXPLAIN, chạy cả ma trận, báo cáo
!REPS={REPS} REF_C2={REF} REF_C3={REF} bash bench/kaggle/run_matrix.sh {MATRIX}

In [ ]:
# Đóng gói toàn bộ kết quả (kể cả log thô) để tải về từ tab Output
!cd bench/results && tar czf /kaggle/working/results.tgz . && ls -lh /kaggle/working/results.tgz

In [ ]:
import glob
from IPython.display import Markdown, display

latest = sorted(glob.glob("bench/results/*/report.md"))[-1]
print(latest)
display(Markdown(open(latest, encoding="utf-8").read()))